In [1]:
# !pip install datasets transformers jiwer torchaudio --quiet

In [2]:
# import os
# import pandas as pd
# from datasets import Dataset, Audio
# from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
# import torch

In [3]:
# DATA_DIR = "/kaggle/input/asr-nepali-openslr-final/asr_nepali/"
# TSV_FILE = os.path.join(DATA_DIR, "utt_spk_text.tsv")

In [4]:
# df = pd.read_csv(TSV_FILE, sep="\t", header=None, names=["utt_id","spk_id","sentence"])

In [5]:
# # This is alternate

# import os

# # Step 1: Get all .flac files in the dataset once
# all_flac_files = []
# for root, dirs, files in os.walk(os.path.join(DATA_DIR, "data")):
#     for f in files:
#         if f.endswith(".flac"):
#             all_flac_files.append(os.path.join(root, f))

# # Step 2: Map utt_id to path
# utt_to_path = {}
# for f in all_flac_files:
#     utt_id = os.path.basename(f).split(".")[0]  # assumes filename starts with utt_id
#     utt_to_path[utt_id] = f

# # Step 3: Assign paths to dataframe
# df["path"] = df["utt_id"].map(utt_to_path)
# df = df.dropna()


In [6]:
# paths = []
# for utt in df.utt_id:
#     found = None
#     for root, dirs, files in os.walk(os.path.join(DATA_DIR,"data")):
#         for f in files:
#             if f.startswith(utt) and f.endswith(".flac"):
#                 found = os.path.join(root,f)
#                 break
#         if found:
#             break
#     paths.append(found)
# df["path"] = paths
# df = df.dropna()

In [7]:
# # create dataset
# dataset = Dataset.from_pandas(df[["path","sentence"]])
# dataset = dataset.cast_column("path", Audio(sampling_rate=16000))

In [8]:
# processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="ne", task="transcribe")

In [9]:
# def prepare(batch):
#     audio = batch["path"]
#     batch["input_features"] = processor(audio["array"], sampling_rate=16000).input_features[0]
#     batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
#     return batch

In [10]:
# dataset = dataset.map(prepare)
# dataset.set_format(type="torch", columns=["input_features","labels"])

In [11]:
# model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
# model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="ne", task="transcribe")

In [12]:
# # train/test split
# train_test = dataset.train_test_split(test_size=0.1)

In [13]:
# def collate(batch):
#     input_features = torch.stack([x["input_features"] for x in batch])
#     labels = processor.tokenizer.pad({"input_ids":[x["labels"] for x in batch]}, return_tensors="pt").input_ids
#     return {"input_features": input_features, "labels": labels}

In [14]:
# args = Seq2SeqTrainingArguments(
#     output_dir="./whisper-nepali",
#     per_device_train_batch_size=4,
#     per_device_eval_batch_size=4,
#     fp16=True,
#     learning_rate=1e-5,
#     num_train_epochs=5,
#     logging_steps=50,
#     save_strategy="no"
# )

In [15]:
# trainer = Seq2SeqTrainer(
#     model=model,
#     args=args,
#     train_dataset=train_test["train"],
#     eval_dataset=train_test["test"],
#     tokenizer=processor.feature_extractor,
#     data_collator=collate
# )

In [16]:
# trainer.train()

In [17]:
# # demo inference
# sample = dataset[0]
# input_feat = sample["input_features"].unsqueeze(0)
# pred_ids = model.generate(input_feat)
# print("Prediction:", processor.tokenizer.decode(pred_ids[0], skip_special_tokens=True))
# print("Target:", sample["sentence"])

In [18]:
# Install dependencies
!pip -q install --upgrade pip
!pip -q install transformers==4.44.2 datasets==2.20.0 accelerate==0.33.0 evaluate==0.4.2 jiwer==3.0.3 torchaudio librosa soundfile gradio==4.44.0

import os, gc, random, numpy as np, torch, pandas as pd
from datasets import Dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 5.5 MB/s eta 0:00:0000:0100:010m
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.40.3 which is incompatible.
google-colab 1.0.0 requires notebook==6.5.7, but you have notebook 6.5.4 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.3, but you have requests 2.32.4 which is incompatible.
google-colab 1.0.0 requires tornado==6.4.2, but you have tornado 6.5.1 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0

2025-08-18 13:08:19.974891: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755522500.185961      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755522500.245699      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [19]:

# config
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)


MODEL_NAME = "openai/whisper-small"   # or "openai/whisper-base" if OOM
LANGUAGE = "ne"
TASK = "transcribe"
SAMPLE_RATE = 16000
MAX_DURATION_S = 30.0
OUTPUT_DIR = "./results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [20]:
# DATA_PATH = "/kaggle/input/asr-nepali-openslr-final/asr_nepali"

# # Load CSV
# df = pd.read_csv(f"{DATA_PATH}/utt_spk_text.tsv")
# # HuggingFace Dataset
# dataset = Dataset.from_pandas(df)

# # Attach audio files
# dataset = dataset.cast_column("path", Audio(sampling_rate=SAMPLE_RATE))

# # Train/test split (adjust ratio)
# dataset = dataset.train_test_split(test_size=0.1)

# print(dataset)
# print(dataset["train"][0])

In [21]:
import os
import pandas as pd
from datasets import Dataset, Audio

SAMPLE_RATE = 16000
DATA_PATH = "/kaggle/input/asr-nepali-openslr-final/asr_nepali"

# Load TSV (no header, so add names manually)
df = pd.read_csv(f"{DATA_PATH}/utt_spk_text.tsv", sep="\t", names=["utt_id", "spk_id", "sentence"])

# Build correct audio path
# utt_id looks like "0000c0f43b"
# File lives in data/<first two chars of utt_id>/<utt_id>.flac
df["path"] = df["utt_id"].apply(
    lambda x: os.path.join(DATA_PATH, "data", x[:2], f"{x}.flac")
)

# HuggingFace Dataset
dataset = Dataset.from_pandas(df)

# Attach audio column
dataset = dataset.cast_column("path", Audio(sampling_rate=SAMPLE_RATE))

# Train/test split
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(dataset)
print(dataset["train"][0])


DatasetDict({
    train: Dataset({
        features: ['utt_id', 'spk_id', 'sentence', 'path'],
        num_rows: 142114
    })
    test: Dataset({
        features: ['utt_id', 'spk_id', 'sentence', 'path'],
        num_rows: 15791
    })
})
{'utt_id': '3adba6fb7e', 'spk_id': '5c4b9', 'sentence': 'सही किसिमको योजना', 'path': {'path': '/kaggle/input/asr-nepali-openslr-final/asr_nepali/data/3a/3adba6fb7e.flac', 'array': array([ 0.00045776,  0.00033569,  0.00036621, ..., -0.00018311,
       -0.00018311, -0.00036621]), 'sampling_rate': 16000}}


In [22]:
# processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)

# def prepare_batch(batch):
#     audio = batch["path"]["array"]
#     inputs = processor.feature_extractor(audio, sampling_rate=SAMPLE_RATE, return_tensors="pt")
#     with processor.as_target_tokenizer():
#         labels = processor.tokenizer(batch["text"])
#     batch["input_features"] = inputs["input_features"][0]
#     batch["labels"] = labels["input_ids"]
#     return batch

# dataset = dataset.map(prepare_batch, remove_columns=dataset["train"].column_names, num_proc=2)
# dataset.set_format(type="torch")

In [24]:
# from datasets import Audio, load_dataset
# from transformers import WhisperProcessor

# MODEL_NAME = "openai/whisper-small"
# LANGUAGE = "ne"
# TASK = "transcribe"
# SAMPLE_RATE = 16000

# processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)

# # Load dataset from folder
# dataset = load_dataset("audiofolder", data_dir="/kaggle/input/asr-nepali-openslr-final/asr_nepali")

# # Inspect to find correct text column
# print(dataset["train"].column_names)

# # Assuming transcript column is called "sentence"
# TEXT_COLUMN = "sentence"
# AUDIO_COLUMN = "path"

# # Decode audio
# dataset = dataset.cast_column(AUDIO_COLUMN, Audio(sampling_rate=SAMPLE_RATE))

# def prepare_batch(batch):
#     # Audio
#     audio = batch[AUDIO_COLUMN]["array"]
#     inputs = processor.feature_extractor(audio, sampling_rate=SAMPLE_RATE, return_tensors="pt")

#     # Text
#     labels = processor.tokenizer(
#         batch[TEXT_COLUMN],
#         return_tensors="pt"
#     )

#     batch["input_features"] = inputs.input_features[0]
#     batch["labels"] = labels.input_ids[0]
#     return batch

# # Map
# dataset = dataset.map(
#     prepare_batch,
#     remove_columns=dataset["train"].column_names,
#     num_proc=2
# )

# dataset.set_format(type="torch", columns=["input_features", "labels"])


In [25]:
import os
import pandas as pd
from datasets import Dataset, Audio

SAMPLE_RATE = 16000
DATA_PATH = "/kaggle/input/asr-nepali-openslr-final/asr_nepali"

# Load TSV (utt_id, spk_id, sentence)
df = pd.read_csv(
    f"{DATA_PATH}/utt_spk_text.tsv",
    sep="\t",
    names=["utt_id", "spk_id", "sentence"]
)

# Build audio path for each utt_id
df["path"] = df["utt_id"].apply(
    lambda x: os.path.join(DATA_PATH, "data", x[:2], f"{x}.flac")
)

# Convert to HuggingFace Dataset
dataset = Dataset.from_pandas(df)

# Attach audio column with decoding
dataset = dataset.cast_column("path", Audio(sampling_rate=SAMPLE_RATE))

# Split into train and test
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(dataset)
print(dataset["train"][0])


DatasetDict({
    train: Dataset({
        features: ['utt_id', 'spk_id', 'sentence', 'path'],
        num_rows: 142114
    })
    test: Dataset({
        features: ['utt_id', 'spk_id', 'sentence', 'path'],
        num_rows: 15791
    })
})
{'utt_id': '3adba6fb7e', 'spk_id': '5c4b9', 'sentence': 'सही किसिमको योजना', 'path': {'path': '/kaggle/input/asr-nepali-openslr-final/asr_nepali/data/3a/3adba6fb7e.flac', 'array': array([ 0.00045776,  0.00033569,  0.00036621, ..., -0.00018311,
       -0.00018311, -0.00036621]), 'sampling_rate': 16000}}


In [26]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: WhisperProcessor
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        with self.processor.as_target_tokenizer():
            labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

In [27]:
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)
model.config.suppress_tokens = []
model.config.use_cache = False
model.gradient_checkpointing_enable()
if torch.cuda.is_available(): model.to("cuda")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [29]:
!pip install --upgrade evaluate jiwer


  Attempting uninstall: jiwer
    Found existing installation: jiwer 3.0.3
    Uninstalling jiwer-3.0.3:
      Successfully uninstalled jiwer-3.0.3
  Attempting uninstall: evaluate
    Found existing installation: evaluate 0.4.2
    Uninstalling evaluate-0.4.2:
      Successfully uninstalled evaluate-0.4.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [evaluate]


In [38]:
# wer_metric = evaluate.load("wer")

# def compute_metrics(pred):
#     pred_ids = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
#     pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
#     label_ids = pred.label_ids
#     label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
#     label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
#     return {"wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str)}

import jiwer

def compute_metrics(pred):
    pred_ids = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = jiwer.wer(label_str, pred_str)
    return {"wer": 100 * wer}




training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    num_train_epochs=3,
    evaluation_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    predict_with_generate=True,
    fp16=True,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    # tokenizer=processor.feature_extractor,
    tokenizer=processor.tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/asr-nepali-openslr-final/asr_nepali/data/f8/f86721dd6a.flac'

In [39]:
print(df["utt_id"].head(10).tolist())

['4aa1fdca33', '86e521554a', 'c96b2b1afb', '2f7ab41953', 'c290e5124b', '1d7f225bd0', '5e766d189e', '8190842174', 'b6c37578d2', 'c40938bd06']


In [ ]:
import librosa

def transcribe_file(path, model_dir=OUTPUT_DIR):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    proc = WhisperProcessor.from_pretrained(model_dir, language=LANGUAGE, task=TASK)
    mdl = WhisperForConditionalGeneration.from_pretrained(model_dir).to(device)
    mdl.config.forced_decoder_ids = proc.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)
    audio, sr = librosa.load(path, sr=SAMPLE_RATE)
    inputs = proc.feature_extractor(audio, sampling_rate=SAMPLE_RATE, return_tensors="pt").to(device)
    with torch.no_grad(): pred_ids = mdl.generate(**inputs)
    return proc.batch_decode(pred_ids, skip_special_tokens=True)[0]

In [ ]:
import gradio as gr

def _transcribe_gradio(audio, model_dir=OUTPUT_DIR):
    if audio is None: return ""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    proc = WhisperProcessor.from_pretrained(model_dir, language=LANGUAGE, task=TASK)
    mdl = WhisperForConditionalGeneration.from_pretrained(model_dir).to(device)
    mdl.config.forced_decoder_ids = proc.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)

    if isinstance(audio, tuple):  # (sr, data)
        sr, data = audio
    elif isinstance(audio, str):  # filepath
        import librosa
        data, sr = librosa.load(audio, sr=SAMPLE_RATE)
    else:
        return "Unsupported audio"

    if sr != SAMPLE_RATE:
        import librosa
        data = librosa.resample(np.asarray(data), orig_sr=sr, target_sr=SAMPLE_RATE)

    inputs = proc.feature_extractor(data, sampling_rate=SAMPLE_RATE, return_tensors="pt").to(device)
    with torch.no_grad(): pred_ids = mdl.generate(**inputs)
    return proc.batch_decode(pred_ids, skip_special_tokens=True)[0]

demo = gr.Interface(
    fn=_transcribe_gradio,
    inputs=gr.Audio(sources=["microphone", "upload"], type="numpy"),
    outputs=gr.Textbox(label="Transcription"),
    title="Nepali ASR (Whisper)",
    description="Upload or record Nepali audio to get transcription"
)

demo.launch(share=False)